# Matryoshka Embeddings From Scratch

Wiki reference for [Matryoshka Embeddings (MRL)](https://ml-viz-ruby.vercel.app/wiki/matryoshka-embeddings).

> Colab: **File → Save a copy in Drive** before editing.

**The problem it solves.** An embedding model normally fixes one output dimension for
life. Choose 3072 and every vector costs 12 kB and every scan multiplies out 3072
coordinates; choose 256 and you capped quality before measuring it. Chopping a normal
embedding down does not work, because nothing in a standard contrastive loss cares
*where* in the vector the information sits.

**The core idea.** Train so that the **first `m` coordinates are themselves a usable
embedding**, for every `m` in a set of nesting granularities (8, 16, 32, … , d). One
forward pass, one set of weights, `|M|` copies of the same loss applied to progressively
shorter prefixes. At inference you truncate to whatever width you can afford, renormalise,
and the cosine metric still means what it meant.

**Where it shows up.** OpenAI's `text-embedding-3-*` `dimensions` parameter, Nomic Embed
v1.5 (any width from 64 to 768), and `sentence-transformers`' `MatryoshkaLoss`.

In this notebook we build a Matryoshka encoder from scratch in NumPy (including the
InfoNCE gradient, checked against finite differences), reproduce the worked loss trace
from the wiki page exactly, measure the information front-loading that MRL creates, and
build the adaptive shortlist-then-rerank cascade that is the real reason MRL exists.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('dark_background')
C = {'mrl': '#6366f1', 'plain': '#f59e0b', 'grid': '#334155',
     'ok': '#22d3ee', 'bad': '#fb7185', 'txt': '#cbd5e1'}
np.set_printoptions(precision=4, suppress=True)

rng = np.random.default_rng(0)

## 1. From scratch

### 1a. A task with a known answer

To see whether a truncated embedding still works we need ground truth about which pairs
*should* be close. The cheapest honest setup: `K` well-separated Gaussian clusters in a
32-dimensional input space, where "same cluster" is the positive relation. A retrieval hit
is then simply a nearest neighbour with the same cluster label.

The encoder is a single linear map `W: R^32 -> R^16` followed by L2 normalisation. Linear
is deliberate. It has no capacity to hide what it is doing, so any front-loading of
information into the early coordinates has to come from the *objective*, not from depth.

In [ ]:
D_IN, D_OUT, K = 32, 16, 16      # input dim, embedding dim, number of classes
NEST = [2, 4, 8, 16]             # the nesting granularities M
TAU = 0.07                       # InfoNCE temperature

CENTERS = rng.normal(size=(K, D_IN))     # one fixed set of class centers

def make_data(n_per, seed):
    """n_per points around each of the K centers. Shapes: X (K*n_per, 32), y (K*n_per,)."""
    r = np.random.default_rng(seed)
    X = np.vstack([CENTERS[k] + 0.85 * r.normal(size=(n_per, D_IN)) for k in range(K)])
    y = np.concatenate([np.full(n_per, k) for k in range(K)])
    return X, y

def l2(v, eps=1e-9):
    """Row-wise L2 normalisation."""
    return v / (np.linalg.norm(v, axis=-1, keepdims=True) + eps)

X_tr, y_tr = make_data(150, 10)      # training pool
X_db, y_db = make_data(200, 11)      # the corpus we retrieve from
X_q,  y_q  = make_data(40,  12)      # held-out queries

print('train', X_tr.shape, ' corpus', X_db.shape, ' queries', X_q.shape)

### 1b. InfoNCE and its gradient

For a batch of `B` anchor/positive pairs with unit-norm rows `A` and `P`, InfoNCE scores
every anchor against every positive in the batch and asks that the diagonal win:

$$S = \frac{A P^\top}{\tau}, \qquad \mathcal{L} = -\frac{1}{B}\sum_i \log \frac{e^{S_{ii}}}{\sum_j e^{S_{ij}}}$$

The gradient is the classic softmax-cross-entropy one, `softmax(S_i) - onehot(i)`, pushed
back through the two matrix products:

$$\frac{\partial \mathcal{L}}{\partial S_{ij}} = \frac{1}{B}\left(\mathrm{softmax}(S_i)_j - \delta_{ij}\right),
\qquad \frac{\partial \mathcal{L}}{\partial A} = \frac{1}{\tau}\frac{\partial \mathcal{L}}{\partial S} P,
\qquad \frac{\partial \mathcal{L}}{\partial P} = \frac{1}{\tau}\frac{\partial \mathcal{L}}{\partial S}^\top A$$

The normalisation step needs its own backward pass. For $\hat{u} = u / \lVert u \rVert$ the
Jacobian is the projection that removes the radial component, because moving *along* $u$
does not change $\hat{u}$:

$$\frac{\partial \hat{u}}{\partial u} = \frac{1}{\lVert u \rVert}\left(I - \hat{u}\hat{u}^\top\right)$$

In [ ]:
def infonce(A, P, tau):
    """A, P: (B, m) with unit rows. Returns (loss, dL/dA, dL/dP)."""
    B = A.shape[0]
    S = A @ P.T / tau
    S = S - S.max(axis=1, keepdims=True)          # stabilise before exp
    e = np.exp(S)
    sm = e / e.sum(axis=1, keepdims=True)
    loss = -np.mean(np.log(sm[np.arange(B), np.arange(B)] + 1e-12))
    dS = (sm - np.eye(B)) / B
    return loss, (dS @ P) / tau, (dS.T @ A) / tau

def norm_backward(u, g):
    """u: (B, m) raw rows; g: dL/d(u/||u||). Returns dL/du."""
    n = np.linalg.norm(u, axis=1, keepdims=True) + 1e-9
    uh = u / n
    return (g - uh * np.sum(g * uh, axis=1, keepdims=True)) / n

### 1c. Check the gradient before trusting it

Hand-derived gradients are where from-scratch implementations go wrong silently: a wrong
gradient still trains, just to a worse place. Compare the analytic gradient of
`loss(normalise(U), normalise(V))` against a central finite difference on a few random
coordinates.

In [ ]:
r = np.random.default_rng(3)
U, V = r.normal(size=(5, 6)), r.normal(size=(5, 6))

def full_loss(U, V):
    return infonce(l2(U), l2(V), TAU)[0]

_, dA, _ = infonce(l2(U), l2(V), TAU)
analytic = norm_backward(U, dA)

h, worst = 1e-6, 0.0
for i, j in [(0, 0), (1, 3), (2, 5), (4, 2), (3, 1)]:
    Up, Um = U.copy(), U.copy()
    Up[i, j] += h
    Um[i, j] -= h
    numeric = (full_loss(Up, V) - full_loss(Um, V)) / (2 * h)
    worst = max(worst, abs(numeric - analytic[i, j]))
    print(f'dL/dU[{i},{j}]  analytic {analytic[i, j]: .6f}   numeric {numeric: .6f}')

print(f'\nmax abs difference: {worst:.2e}')
assert worst < 1e-6, 'gradient mismatch'

**What to notice.** The analytic and numeric gradients agree to ~1e-8. Everything below
rests on this cell, so it is worth the twelve lines.

### 1d. The MRL loss: the same loss, `|M|` times, on prefixes

This is the entire idea in one loop. Encode the batch **once**, then for each granularity
`m` slice the first `m` coordinates, renormalise the slice, and accumulate the same
InfoNCE. The gradient at granularity `m` touches only the first `m` coordinates, so
coordinate 0 receives gradient from all four terms and coordinate 15 from one. That
asymmetry is the whole mechanism.

$$\mathcal{L}_{\text{MRL}} = \sum_{m \in \mathcal{M}} c_m \cdot \mathcal{L}\!\left(\frac{z_{1:m}}{\lVert z_{1:m} \rVert}\right)$$

Passing `nest=[16]` gives back plain InfoNCE, which is exactly the baseline we want to
compare against: same data, same optimiser, same steps, one term instead of four.

In [ ]:
def train(nest, X, y, steps=600, batch=96, lr=0.5, seed=1, weights=None):
    """Train a linear encoder with the MRL loss over granularities `nest`."""
    r = np.random.default_rng(seed)
    W = r.normal(size=(D_IN, D_OUT)) / np.sqrt(D_IN)
    by_class = [np.where(y == k)[0] for k in range(K)]
    c = weights if weights is not None else {m: 1.0 for m in nest}
    hist = []

    for _ in range(steps):
        cls = r.choice(K, size=batch, replace=True)
        ia = np.array([r.choice(by_class[k]) for k in cls])
        ip = np.array([r.choice(by_class[k]) for k in cls])
        # keep one pair per class so in-batch negatives are genuinely negative
        _, first = np.unique(cls, return_index=True)
        ia, ip = ia[first], ip[first]

        Xa, Xp = X[ia], X[ip]
        Za, Zp = Xa @ W, Xp @ W                  # (B, 16) raw embeddings
        gZa, gZp = np.zeros_like(Za), np.zeros_like(Zp)
        total = 0.0

        for m in nest:                            # <-- the Matryoshka loop
            Ua, Up = Za[:, :m], Zp[:, :m]         # slice the prefix
            A, P = l2(Ua), l2(Up)                 # renormalise it
            loss, dA, dP = infonce(A, P, TAU)
            total += c[m] * loss
            gZa[:, :m] += c[m] * norm_backward(Ua, dA)
            gZp[:, :m] += c[m] * norm_backward(Up, dP)

        W -= lr * (Xa.T @ gZa + Xp.T @ gZp)
        hist.append(total)
    return W, hist

### 1e. Train both models

Identical everything except the set of granularities.

In [ ]:
W_mrl,   h_mrl   = train(NEST, X_tr, y_tr)
W_plain, h_plain = train([16], X_tr, y_tr)

print(f'final training loss   MRL (4 terms) {h_mrl[-1]:.4f}   plain (1 term) {h_plain[-1]:.4f}')
print('(not comparable directly: the MRL number is a sum of four losses)')

## 2. Cross-check: reproduce the wiki page's worked trace

The wiki page walks one MRL loss by hand on a 3-pair batch with `d=4`, `M={2,4}`,
`tau=0.2`. If the implementation above is right it must reproduce those numbers exactly.

In [ ]:
Za_t = np.array([[0.90,  0.20,  0.30, -0.10],
                 [0.80,  0.30, -0.60,  0.40],
                 [0.10, -0.30,  0.90,  0.20]])
Zp_t = np.array([[0.80,  0.40,  0.10,  0.20],
                 [0.85,  0.20, -0.50,  0.50],
                 [0.20, -0.20,  0.80,  0.40]])
TAU_T = 0.2

total = 0.0
for m in (2, 4):
    A, P = l2(Za_t[:, :m]), l2(Zp_t[:, :m])
    S = A @ P.T
    loss, _, _ = infonce(A, P, TAU_T)
    total += loss
    e = np.exp(S / TAU_T)
    per_row = -np.log(np.diag(e) / e.sum(axis=1))
    print(f'--- m = {m}')
    print('similarity matrix S = A @ P.T')
    print(S)
    print(f'per-row loss {per_row.round(4)}   mean L_{m} = {loss:.4f}\n')

print(f'total MRL loss = {total:.4f}')
print(f'share carried by the m=2 term: {0.5241 / total * 100:.0f}%')

assert np.isclose(total, 0.6945, atol=5e-4)

**What to notice.** At `m = 2` the off-diagonal entry `S[0,1] = 1.000` **beats** the
diagonal `S[0,0] = 0.970`: truncated to two coordinates, pairs 1 and 2 are the same point,
because they differ mostly in coordinate 3. At `m = 4` that coordinate becomes visible and
every diagonal entry dominates its row.

The consequence is the number on the last line: **about 75% of the total loss comes from
the 2-dimensional term**. The coarse granularities are the hard ones, so they dominate the
gradient, so the optimiser spends its capacity making the early coordinates discriminative.
Delete the `m = 2` term and you are back to plain InfoNCE, which is content to leave those
coordinates as noise.

## 3. Visualize it

### 3a. Where does the information sit?

Take unit-normalised corpus embeddings and measure the fraction of squared norm held by
the first `m` coordinates. A model with no preference gives a straight diagonal.

In [ ]:
def cum_energy(W):
    Z = l2(X_db @ W)
    var = (Z ** 2).mean(axis=0)
    return np.cumsum(var) / var.sum()

e_mrl, e_plain = cum_energy(W_mrl), cum_energy(W_plain)
dims = np.arange(1, D_OUT + 1)

fig, ax = plt.subplots(figsize=(7, 4.2))
ax.plot(dims, e_plain, 'o-', color=C['plain'], label='plain InfoNCE')
ax.plot(dims, e_mrl, 'o-', color=C['mrl'], label='MRL  M = {2,4,8,16}')
ax.plot(dims, dims / D_OUT, '--', color=C['grid'], label='uniform (no preference)')
for m in NEST[:-1]:
    ax.axvline(m, color=C['grid'], lw=0.8, alpha=0.6)
ax.set_xlabel('first m coordinates'); ax.set_ylabel('share of squared norm')
ax.set_title('MRL front-loads information into the early coordinates')
ax.legend(frameon=False); ax.grid(alpha=0.15)
plt.tight_layout(); plt.show()

for m in (2, 4, 8):
    print(f'first {m:2d} coords:  plain {e_plain[m-1]:.1%}   MRL {e_mrl[m-1]:.1%}')

**What to notice.** The plain model tracks the uniform diagonal almost exactly: four of
sixteen coordinates hold about 22% of the energy, which is what "the objective never asked"
looks like. The MRL curve bends hard at the granularities it was trained on. Chopping the
plain model at 4 dimensions throws away four fifths of the representation; chopping the MRL
model there keeps nearly two thirds.

### 3b. Does the front-loading buy retrieval quality?

Energy is a proxy. The thing we actually care about is whether a truncated vector still
retrieves the right neighbour.

In [ ]:
def recall_at1(W, m):
    Q, Db = l2((X_q @ W)[:, :m]), l2((X_db @ W)[:, :m])
    nn = np.argmax(Q @ Db.T, axis=1)
    return float(np.mean(y_db[nn] == y_q))

widths = [2, 4, 8, 16]
r_mrl   = [recall_at1(W_mrl, m) for m in widths]
r_plain = [recall_at1(W_plain, m) for m in widths]

fig, ax = plt.subplots(figsize=(7, 4.2))
ax.plot(widths, r_plain, 'o-', color=C['plain'], label='plain InfoNCE')
ax.plot(widths, r_mrl, 'o-', color=C['mrl'], label='MRL')
ax.set_xscale('log', base=2); ax.set_xticks(widths); ax.set_xticklabels(widths)
ax.set_xlabel("query dimension d'")
ax.set_ylabel('recall@1')
ax.set_title('Quality under truncation')
ax.legend(frameon=False); ax.grid(alpha=0.15)
plt.tight_layout(); plt.show()

print(f"{'d':>4} {'plain':>8} {'MRL':>8}   delta")
for m, a, b in zip(widths, r_plain, r_mrl):
    print(f'{m:>4} {a:>8.3f} {b:>8.3f}   {b - a:+.3f}')

**What to notice.** Read the first and last rows together, because they are the whole
trade. At full width MRL gives up about 0.3 points. At a quarter width it gains over 30.
That is the bargain: a fraction of a point of peak quality, in exchange for a model that
works at every width you might want to deploy.

Notice also that the gap *closes* as `d'` grows. MRL is not a better model, it is the same
model with its coordinates ordered.

## 4. Truncate, then renormalise

The single most common MRL bug, and it is silent. A prefix of a unit vector has norm below
1, **and by a different amount for every vector**. Skip the renormalisation and your
"cosine similarity" quietly becomes a dot product that rewards documents for keeping more
of their mass in the prefix, which has nothing to do with relevance.

In [ ]:
q  = l2(np.array([[0.60, 0.60, 0.40, 0.34]]))[0]
d1 = l2(np.array([[0.50, 0.50, 0.60, 0.38]]))[0]
d2 = l2(np.array([[0.72, 0.36, 0.30, 0.51]]))[0]

def show(m, renorm):
    a, b1, b2 = q[:m], d1[:m], d2[:m]
    if renorm:
        a, b1, b2 = a / np.linalg.norm(a), b1 / np.linalg.norm(b1), b2 / np.linalg.norm(b2)
    s1, s2 = a @ b1, a @ b2
    tag = 'renormalised' if renorm else 'raw dot     '
    win = 'd1' if s1 > s2 else 'd2'
    print(f'  m={m}  {tag}   q.d1 = {s1:.4f}   q.d2 = {s2:.4f}   -> winner {win}')

print('ground truth at full width:')
show(4, True)
print('\ntruncated to the first 2 coordinates:')
show(2, False)
show(2, True)
print(f'\nprefix norms: ||d1[:2]|| = {np.linalg.norm(d1[:2]):.4f}   '
      f'||d2[:2]|| = {np.linalg.norm(d2[:2]):.4f}')

**What to notice.** The raw-dot row **flips the ranking**: `d2` wins at 0.6500 against
0.6000, while the full-width answer and the renormalised prefix both pick `d1`. The cause
is on the last line. `d2` keeps 0.806 of its norm in the first two coordinates against
`d1`'s 0.706, so it wins on surviving norm rather than on direction. Dividing that
advantage out restores the correct order.

In library code the two lines are:

```python
v = model.encode(text)[:256]
v = v / np.linalg.norm(v)      # not optional
```

## 5. Adaptive retrieval: the reason MRL exists

Truncating to save memory is the obvious use. The better one is a **cascade**: shortlist
over the whole corpus at low dimension, then rerank the shortlist at full dimension. The
shortlist only has to be *recall*-accurate, and that is exactly what a truncated embedding
is still good at. Deciding *which of these two is closer* is what it is bad at, and that is
what the rerank does.

In [ ]:
def adaptive(W, m_lo, m_hi, k_short):
    """Shortlist k_short candidates at m_lo dims, rerank them at m_hi dims."""
    Q, Db = X_q @ W, X_db @ W
    Q_lo, D_lo = l2(Q[:, :m_lo]), l2(Db[:, :m_lo])
    Q_hi, D_hi = l2(Q[:, :m_hi]), l2(Db[:, :m_hi])

    sims = Q_lo @ D_lo.T                                    # cheap full scan
    cand = np.argpartition(-sims, k_short, axis=1)[:, :k_short]

    hits = 0
    for i in range(len(X_q)):
        c = cand[i]
        best = c[np.argmax(Q_hi[i] @ D_hi[c].T)]            # expensive, but on k_short rows
        hits += int(y_db[best] == y_q[i])
    return hits / len(X_q)

print(f"shortlist at d'=4 only          {recall_at1(W_mrl, 4):.3f}")
for k in (5, 10, 25, 50):
    print(f'shortlist {k:>2} at 4, rerank at 16   {adaptive(W_mrl, 4, 16, k):.3f}')
print(f'full 16-dim exact scan          {recall_at1(W_mrl, 16):.3f}')

**What to notice.** A shortlist of 25 recovers 0.988 of the 0.994 exact ceiling while the
expensive full-width comparison runs on 25 rows instead of 3200. The gap the rerank closes
is precisely the "which of these two" question the truncated vector was bad at.

Now the arithmetic at production scale, which is where the factor becomes interesting.

In [ ]:
N, d, d_lo, k_short = 10_000_000, 3072, 256, 100

full    = N * d
shortlist, rerank = N * d_lo, k_short * d
adaptive_cost = shortlist + rerank

print(f'full exact scan     {full:.3e} multiplications')
print(f'shortlist at {d_lo}    {shortlist:.3e}   ({shortlist / adaptive_cost:.2%} of adaptive cost)')
print(f'rerank top {k_short}       {rerank:.3e}')
print(f'adaptive total      {adaptive_cost:.3e}   -> {full / adaptive_cost:.1f}x cheaper\n')

for label, dim, bytes_per in [('3072-dim fp32', d, 4), ('256-dim fp32', d_lo, 4),
                              ('256-dim int8', d_lo, 1)]:
    print(f'{label:>14} index for {N // 1_000_000}M docs: {N * dim * bytes_per / 1e9:6.1f} GB')

**What to notice.** The rerank is *free*: three hundred thousand multiplications against
two and a half billion. That is the structural point. Once the shortlist is 100 documents
instead of 10 million you can afford any scoring function you like on it, and full-width
cosine is the cheapest thing you might reach for.

Memory moves by the same 12×, and memory is usually the binding constraint because the
shortlist index wants to be resident in RAM. The 256-dim index fits on one machine; the
full one does not. The full vectors can live on disk, because a query only ever fetches 100
of them.

The published result has the same shape at real scale: MRL's shortlist-then-rerank matched
baseline mAP@10 on ImageNet-4K at **128× fewer FLOPs and 14× faster wall-clock**
(Kusupati et al., NeurIPS 2022).

## 6. Tradeoffs and when to use it

| | MRL | A second, smaller model | PCA on trained embeddings |
|---|---|---|---|
| Models to maintain | 1 | 2 | 1 |
| Re-embed to change dimension | no | yes | no |
| Full-width quality | ~0.3 pt lower (measured above) | unchanged | unchanged |
| Low-width quality | best | comparable | worse |
| Needs retraining to adopt | yes | yes | no |

**Cost.** One forward pass, `|M|` cheap loss terms on a `B x B` matrix. The extra
similarity work is $\sum_m m \approx 2d$ columns instead of $d$, which is a few percent of
step time next to the encoder itself.

**Key hyperparameters.**

- `M`, the granularities: powers of two down from `d`, stopping where quality collapses on
  your own data. Unlisted widths still work, but they are untested and the quality curve
  has its dents between the trained points.
- `c_m`, the per-granularity weights: all 1 by default. Weight the small `m` up when you
  know most traffic is served truncated, and pay for it at full width.
- `tau`, temperature: unchanged from ordinary InfoNCE, typically 0.05 to 0.1.

**Failure modes.**

1. Forgetting to renormalise after truncating (section 4).
2. Truncating a model that was never trained with MRL. Check the model card.
3. Mixing widths, or models, inside one index. Not comparable.
4. Expecting a speedup without rebuilding the ANN index over the prefixes. Truncation is
   free for storage and for a brute-force scan, not for an index built at full width.

**Why PCA loses.** PCA picks directions of maximum variance, which is a *reconstruction*
criterion. MRL picks coordinates that maximise the *contrastive* objective at that width.
Variance and discriminability correlate but are not the same thing, and the gap opens
exactly on the hard pairs.

## ✏️ Your turn

### Exercise 1 — buy low-width quality with `c_m`

The weights `c_m` are the knob for "most of my traffic is truncated". Train an MRL encoder
that weights the small granularities up and measure what it costs at full width.

Train with `weights={2: 4.0, 4: 4.0, 8: 1.0, 16: 1.0}` and compare recall@1 at every width
against the equal-weights `W_mrl` from section 1e.

In [ ]:
# TODO(you): train a second MRL encoder with the small granularities weighted up,
# then print recall@1 at d' = 2, 4, 8, 16 for both it and W_mrl.
#
# Hints:
#   - train(NEST, X_tr, y_tr, weights={...}) takes the dict directly
#   - recall_at1(W, m) is defined in section 3b

W_weighted = ...   # <- your call to train(...)

# for m in [2, 4, 8, 16]:
#     print(m, recall_at1(W_mrl, m), recall_at1(W_weighted, m))

In [ ]:
#@title Solution (run to reveal)
W_weighted, _ = train(NEST, X_tr, y_tr, weights={2: 4.0, 4: 4.0, 8: 1.0, 16: 1.0})

print(f"{'d':>4} {'equal':>8} {'weighted':>10}   delta")
for m in [2, 4, 8, 16]:
    a, b = recall_at1(W_mrl, m), recall_at1(W_weighted, m)
    print(f'{m:>4} {a:>8.3f} {b:>10.3f}   {b - a:+.3f}')

print('\nThe knob works, and it is neither free nor monotone. d=4 gains about 4 points,')
print('full width pays about 4, and d=2 barely moves: at 640 queries a 0.013 swing is')
print('roughly eight of them. Weighting buys quality at the widths the extra weight can')
print('actually reach, so tune c_m against the widths your traffic really uses, and read')
print('your own numbers rather than assuming the curve shifts uniformly.')

### Exercise 2 — find the knee

Section 5 used a shortlist width of 4 and a shortlist size of 25 without justifying either.
The right pair is corpus-specific and has to be swept.

Sweep `m_lo` over `[2, 4, 8]` and `k_short` over `[5, 10, 25, 50, 100]`, print the recall@1
grid, and find the cheapest configuration that stays within 0.01 of the full 16-dim exact
scan. Cost is `len(X_db) * m_lo + k_short * 16` multiplications per query.

In [ ]:
# TODO(you): sweep (m_lo, k_short), print recall and per-query cost, and pick the
# cheapest configuration within 0.01 of the exact ceiling.
#
# Hints:
#   - adaptive(W_mrl, m_lo, 16, k_short) gives the recall
#   - the exact ceiling is recall_at1(W_mrl, 16)

ceiling = recall_at1(W_mrl, 16)
# for m_lo in [2, 4, 8]:
#     for k_short in [5, 10, 25, 50, 100]:
#         ...

In [ ]:
#@title Solution (run to reveal)
ceiling = recall_at1(W_mrl, 16)
best = None

print(f'exact 16-dim ceiling: {ceiling:.3f}\n')
print(f"{'m_lo':>5} {'k':>5} {'recall':>8} {'cost/query':>12}")
for m_lo in [2, 4, 8]:
    for k_short in [5, 10, 25, 50, 100]:
        rec = adaptive(W_mrl, m_lo, 16, k_short)
        cost = len(X_db) * m_lo + k_short * D_OUT
        flag = ''
        if rec >= ceiling - 0.01:
            flag = ' <- within 0.01'
            if best is None or cost < best[2]:
                best = (m_lo, k_short, cost, rec)
        print(f'{m_lo:>5} {k_short:>5} {rec:>8.3f} {cost:>12,}{flag}')

print(f'\ncheapest configuration within 0.01 of exact: m_lo={best[0]}, k={best[1]}, '
      f'cost {best[2]:,} ({len(X_db) * D_OUT / best[2]:.1f}x cheaper than exact)')
print('The knee is at a smaller m_lo than you would pick without reranking, because the')
print('rerank is covering for the shortlist. Sweep the two together, never separately.')

## Key takeaways

- **The constraint, not the architecture.** MRL changes only the loss: apply the same
  objective to every prefix `z[:m]` of the same shared encoder output. No extra heads, no
  second model, one forward pass.
- **Early coordinates get more gradient.** Coordinate 0 receives a term from every
  granularity, coordinate `d-1` from one. In the worked trace the 2-dim term carried 75% of
  the loss, which is why the optimiser front-loads the representation.
- **Truncate, then renormalise.** A prefix of a unit vector is not a unit vector, and not
  by the same factor for every vector. Skipping this silently reranks by surviving norm.
- **The payoff is the cascade.** Shortlist cheap at low width, rerank on 100 rows at full
  width. In the arithmetic above that is 12× fewer multiplications and a 122.9 GB index
  turned into 10.2 GB, at 0.988 of exact recall.
- **It is not free.** About 0.3 points of full-width quality in the run here. Skip MRL when
  you serve exactly one dimension and it already fits in memory.

**Next:** [Vector Databases & ANN Indexes](https://ml-viz-ruby.vercel.app/wiki/vector-databases)
and [HNSW](https://ml-viz-ruby.vercel.app/wiki/hnsw) for the index to build over the
truncated prefixes, and
[Training Embedding Models](https://ml-viz-ruby.vercel.app/courses/nlp/07-training-embedding-models)
for the contrastive objectives MRL wraps.